# MAI391 — Mathematical Foundation of Multiple Linear Regression
**Học phần:** MAI391 — Mathematics for Machine Learning  
**Dự án:** Red Wine Quality Prediction using Multiple Linear Regression  
**Thành viên thực hiện:** Dao (Member 3 — Mathematical Foundation)  
**Slide phụ trách:** Slide 5: Mathematical Foundation  

---
## Mục tiêu của Notebook
Notebook này trình bày chi tiết nền tảng toán học cho bài toán dự đoán chất lượng rượu vang đỏ bằng Hồi quy tuyến tính đa biến (Multiple Linear Regression), bao gồm:
1. **Biểu diễn toán học bằng Ma trận & Vector:** Định nghĩa không gian dữ liệu $X \in \mathbb{R}^{n \times d}$, vector mục tiêu $y \in \mathbb{R}^n$, vector trọng số $w$ và hệ số chặn $b$.
2. **Hàm mất mát Mean Squared Error (MSE):** Công thức vô hướng và chuẩn Euclid ($L_2$ norm).
3. **Phương pháp Bình phương tối thiểu (Least Squares) & Phương trình chuẩn (Normal Equation):** Thiết lập ma trận thiết kế mở rộng $X_b$, đạo hàm ma trận và nghiệm tối ưu $\theta^* = (X_b^T X_b)^{-1} X_b^T y$.
4. **Kết nối toán học với Scikit-Learn:** Ánh xạ tường minh giữa công thức toán học với các thuộc tính trong code (`model.coef_`, `model.intercept_`, `model.predict()`).
5. **Thực nghiệm kiểm chứng số học:** So sánh độ chính xác giữa nghiệm giải tích đóng (Normal Equation) và mô hình Scikit-learn trên tập dữ liệu chuẩn `data/train.csv`.

In [1]:
from pathlib import Path
import json
import numpy as np
import pandas as pd
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score

# Định dạng hiển thị số thực
np.set_printoptions(precision=6, suppress=True)
pd.set_option('display.float_format', lambda x: '%.6f' % x)

# Tự động xác định Project Root để hoạt động đồng bộ:
# Dù chạy từ repo root (jupyter notebook .) hay từ thư mục notebooks/
def find_project_root(start: Path) -> Path:
    start = start.resolve()
    for parent in (start, *start.parents):
        if (parent / "data").is_dir() and (parent / "notebooks").is_dir():
            return parent
    if start.name == "notebooks":
        return start.parent
    if (start / "notebooks").is_dir():
        return start
    raise FileNotFoundError("Không tìm thấy project root. Vui lòng mở notebook từ repo.")

ROOT = find_project_root(Path.cwd())
DATA_DIR = ROOT / "data"
REPORTS_DIR = ROOT / "reports"

print("Project root:", ROOT)
print("Data directory:", DATA_DIR)
print("Reports directory:", REPORTS_DIR)

Project root: C:\Users\Laptop\Downloads\New folder (2)\repo
Data directory: C:\Users\Laptop\Downloads\New folder (2)\repo\data
Reports directory: C:\Users\Laptop\Downloads\New folder (2)\repo\reports


---
## 1. Biểu diễn toán học bằng Ma trận & Vector (Data Representation)

Trong bài toán Wine Quality Prediction:
- Số lượng mẫu huấn luyện: $n = 1279$
- Số lượng đặc trưng hóa lý (features): $d = 11$

### 1.1. Ma trận đặc trưng $X$ và Vector mục tiêu $y$
Ma trận đầu vào $X \in \mathbb{R}^{n \times 11}$ có cấu trúc:
$$
X = \begin{bmatrix}
x_{1,1} & x_{1,2} & \dots & x_{1,11} \\
x_{2,1} & x_{2,2} & \dots & x_{2,11} \\
\vdots & \vdots & \ddots & \vdots \\
x_{n,1} & x_{n,2} & \dots & x_{n,11}
\end{bmatrix} \in \mathbb{R}^{n \times 11}
$$

Mỗi hàng $x_i^T = [x_{i,1}, x_{i,2}, \dots, x_{i,11}]$ đại diện cho 1 mẫu rượu vang đỏ với 11 chỉ số hóa lý.

Vector nhãn thực tế $y \in \mathbb{R}^{n}$ (điểm chất lượng `quality`):
$$
y = \begin{bmatrix} y_1 \\ y_2 \\ \vdots \\ y_n \end{bmatrix} \in \mathbb{R}^n
$$

### 1.2. Vector trọng số $w$ và Hệ số chặn (Bias / Intercept) $b$
- Vector trọng số (weights): $w = [w_1, w_2, \dots, w_{11}]^T \in \mathbb{R}^{11}$, trong đó mỗi trọng số $w_j$ biểu thị sự thay đổi kỳ vọng của biến mục tiêu khi đặc trưng thứ $j$ thay đổi một đơn vị đo tương ứng, với điều kiện giữ nguyên các đặc trưng khác.
- Hệ số chặn (intercept / bias): $b \in \mathbb{R}$. Đây là hằng số toán học dịch chuyển siêu phẳng hồi quy, biểu thị giá trị dự đoán lý thuyết khi mọi đặc trưng đầu vào cùng bằng $0$ (tại vector zero $\mathbf{0}_{11}$). Trong thực tế hóa lý của rượu vang, một mẫu rượu không thể có nồng độ cồn, độ pH hay tỷ trọng bằng 0; vì vậy $b$ là giá trị toán học của siêu phẳng chứ không đại diện cho một trạng thái rượu thực tế.

Công thức dự đoán điểm chất lượng cho mẫu thứ $i$:
$$
\hat{y}_i = b + w_1 x_{i,1} + w_2 x_{i,2} + \dots + w_{11} x_{i,11} = x_i^T w + b
$$

Trên toàn bộ tập dữ liệu (dạng vector hóa):
$$
\hat{y} = X w + b \mathbf{1}_n \in \mathbb{R}^n
$$
(với $\mathbf{1}_n$ là vector cột toàn số 1 độ dài $n$).

In [2]:
# Đọc dữ liệu từ đường dẫn project root đã xác định
train_df = pd.read_csv(DATA_DIR / "train.csv")
test_df = pd.read_csv(DATA_DIR / "test.csv")

features = [col for col in train_df.columns if col != 'quality']
print(f"Số lượng đặc trưng (d): {len(features)}")
print("Danh sách 11 đặc trưng theo đúng thứ tự:")
for idx, f in enumerate(features, 1):
    print(f"  {idx:2d}. {f}")

X_train = train_df[features].values
y_train = train_df['quality'].values
X_test = test_df[features].values
y_test = test_df['quality'].values

print(f"\nKích thước ma trận X_train: {X_train.shape} (n = {X_train.shape[0]}, d = {X_train.shape[1]})")
print(f"Kích thước vector y_train: {y_train.shape}")

Số lượng đặc trưng (d): 11
Danh sách 11 đặc trưng theo đúng thứ tự:
   1. fixed acidity
   2. volatile acidity
   3. citric acid
   4. residual sugar
   5. chlorides
   6. free sulfur dioxide
   7. total sulfur dioxide
   8. density
   9. pH
  10. sulphates
  11. alcohol

Kích thước ma trận X_train: (1279, 11) (n = 1279, d = 11)
Kích thước vector y_train: (1279,)


---
## 2. Hàm mất mát Mean Squared Error (MSE Loss Function)

Để tìm mô hình tốt nhất, ta cần định lượng sai số giữa giá trị thực tế $y$ và giá trị dự đoán $\hat{y}$.

### 2.1. Dạng tính tổng vô hướng:
$$
J(w, b) = \text{MSE} = \frac{1}{n} \sum_{i=1}^{n} (y_i - \hat{y}_i)^2 = \frac{1}{n} \sum_{i=1}^{n} \left( y_i - (x_i^T w + b) \right)^2
$$

### 2.2. Dạng chuẩn Đại số tuyến tính:
Sử dụng chuẩn Euclidean $L_2$ bình phương:
$$
J(w, b) = \frac{1}{n} \| y - \hat{y} \|_2^2 = \frac{1}{n} \| y - (X w + b \mathbf{1}_n) \|_2^2
$$

**Ý nghĩa toán học:**
1. Bình phương sai số đảm bảo sai số âm hay dương đều mang giá trị dương, không bị triệt tiêu lẫn nhau.
2. Phạt rất nặng các điểm có độ lệch lớn (outliers).
3. **Tính lồi & Điều kiện nghiệm duy nhất:**
   - Ma trận Hessian của hàm mất mát $\nabla_\theta^2 J(\theta) = \frac{2}{n} X_b^T X_b$ luôn là ma trận nửa xác định dương (positive semi-definite), vì vậy (\theta)$ là một hàm lồi (convex function). Tính chất lồi đảm bảo mọi nghiệm của phương trình gradient $\nabla_\theta J = \mathbf{0}$ đều là điểm cực tiểu toàn cục.
   - **Lưu ý toán học quan trọng:** Tính lồi thuần túy chưa đủ để khẳng định nghiệm là duy nhất (nếu các đặc trưng phụ thuộc tuyến tính thì hệ phương trình chuẩn có vô số nghiệm tối ưu). Nghiệm cực tiểu toàn cục chỉ **duy nhất** khi và chỉ khi ma trận thiết kế $ đạt đủ hạng cột (**full column rank**, $\text{rank}(X_b) = d + 1 = 12$). Khi đó ma trận Gram ^T X_b$ xác định dương chặt (strictly positive definite) và khả nghịch, đảm bảo tồn tại duy nhất nghiệm giải tích $\theta^* = (X_b^T X_b)^{-1} X_b^T y$.


---
## 3. Least Squares & Phương trình chuẩn (Normal Equation)

### 3.1. Gộp Bias vào Ma trận Thiết kế ($X_b$)
Để biến đổi bài toán về dạng ma trận đồng nhất, ta thêm một cột toàn số 1 vào đầu ma trận $X$:
$$
X_b = [\mathbf{1}_n \quad X] = \begin{bmatrix}
1 & x_{1,1} & \dots & x_{1,11} \\
1 & x_{2,1} & \dots & x_{2,11} \\
\vdots & \vdots & \ddots & \vdots \\
1 & x_{n,1} & \dots & x_{n,11}
\end{bmatrix} \in \mathbb{R}^{n \times (11+1)}
$$

Vector tham số hợp nhất $\theta$:
$$
\theta = \begin{bmatrix} b \\ w_1 \\ \vdots \\ w_{11} \end{bmatrix} \in \mathbb{R}^{12}
$$

Khi đó, mô hình dự đoán trở thành:
$$
\hat{y} = X_b \theta
$$

Hàm mất mát biểu diễn theo chuẩn ma trận:
$$
J(\theta) = \frac{1}{n} (y - X_b \theta)^T (y - X_b \theta)
$$

### 3.2. Đạo hàm ma trận & Tìm nghiệm tối ưu (Derivation)
Khai triển hàm mất mát:
$$
J(\theta) = \frac{1}{n} \left( y^T y - 2 \theta^T X_b^T y + \theta^T X_b^T X_b \theta \right)
$$

Lấy đạo hàm ma trận theo vector $\theta$:
$$
\nabla_\theta J(\theta) = \frac{2}{n} \left( X_b^T X_b \theta - X_b^T y \right)
$$

Để tìm điểm cực tiểu của hàm lồi $J(\theta)$, ta giải phương trình đạo hàm bằng 0:
$$
\nabla_\theta J(\theta) = \mathbf{0} \iff X_b^T X_b \theta = X_b^T y
$$
Phương trình này được gọi là **Phương trình chuẩn (Normal Equation)**.

Khi ma trận $X_b$ có các cột độc lập tuyến tính (full column rank, $\text{rank}(X_b) = 12$), ma trận Gram $(X_b^T X_b)$ khả nghịch:
$$
\theta^* = (X_b^T X_b)^{-1} X_b^T y
$$
Trong đó:
- $\theta^*[0] = b^*$ (Hệ số chặn / Intercept tối ưu)
- $\theta^*[1:] = w^* = [w_1^*, \dots, w_{11}^*]^T$ (Vector trọng số tối ưu ứng với 11 đặc trưng)

In [3]:
# Tính toán trực tiếp nghiệm tối ưu bằng Normal Equation trong NumPy
X_b_train = np.c_[np.ones((X_train.shape[0], 1)), X_train]
X_b_test = np.c_[np.ones((X_test.shape[0], 1)), X_test]

print("Kích thước ma trận thiết kế X_b_train:", X_b_train.shape)

# Kiểm tra điều kiện khả nghịch (Rank của ma trận)
matrix_rank = np.linalg.matrix_rank(X_b_train)
print(f"Rank của ma trận X_b: {matrix_rank} / {X_b_train.shape[1]} (Đầy đủ hạng cột)")

# Nghiệm giải tích đóng (Normal Equation)
theta_optimal = np.linalg.inv(X_b_train.T @ X_b_train) @ X_b_train.T @ y_train

b_math = theta_optimal[0]
w_math = theta_optimal[1:]

print(f"\n--- KẾT QUẢ TÍNH TOÁN BẰNG NORMAL EQUATION ---")
print(f"Intercept b* : {b_math:.6f}")
print("Trọng số w* ứng với 11 đặc trưng:")
for f_name, w_val in zip(features, w_math):
    print(f"  {f_name:25s}: {w_val:12.6f}")

Kích thước ma trận thiết kế X_b_train: (1279, 12)
Rank của ma trận X_b: 12 / 12 (Đầy đủ hạng cột)

--- KẾT QUẢ TÍNH TOÁN BẰNG NORMAL EQUATION ---
Intercept b* : 14.355105
Trọng số w* ứng với 11 đặc trưng:
  fixed acidity            :     0.023085
  volatile acidity         :    -1.001304
  citric acid              :    -0.140821
  residual sugar           :     0.006564
  chlorides                :    -1.806503
  free sulfur dioxide      :     0.005627
  total sulfur dioxide     :    -0.003644
  density                  :   -10.351594
  pH                       :    -0.393688
  sulphates                :     0.841172
  alcohol                  :     0.281890


---
## 4. Kết nối trực tiếp giữa Công thức Toán học và Scikit-Learn

Bảng đối chiếu toàn diện giữa lý thuyết toán học môn MAI391 và thư viện Scikit-learn:

| Khái niệm Toán học | Ký hiệu Toán | Trong Scikit-Learn | Ý nghĩa trong bài toán Wine Quality |
| :--- | :---: | :--- | :--- |
| **Vector trọng số** | $w \in \mathbb{R}^{11}$ | `model.coef_` | Hệ số hồi quy tuyến tính của từng đặc tính hóa lý (cùng thứ tự 11 cột của $X$) |
| **Hệ số chặn / Intercept** | $b \in \mathbb{R}$ | `model.intercept_` | Giá trị toán học tại gốc tọa độ (khi $x = \mathbf{0}$) của siêu phẳng hồi quy |
| **Mô hình dự đoán** | $\hat{y} = X w + b$ | `model.predict(X)` | Phép nhân ma trận - vector cộng bias để ra điểm chất lượng dự đoán |
| **Nghiệm tối ưu** | $\theta^* = (X_b^T X_b)^{-1} X_b^T y$ | `model.fit(X, y)` | Scikit-learn tìm nghiệm Least Squares bằng phân rã giá trị kỳ dị (SVD) |
| **Hàm mất mát** | $\frac{1}{n}\|y - \hat{y}\|_2^2$ | `mean_squared_error(y, y_pred)` | Trung bình bình phương sai số giữa điểm thực tế và điểm dự đoán |

> **Lưu ý phương pháp luận:**
> Trong bài giảng toán học, ta biểu diễn nghiệm giải tích qua ma trận nghịch đảo $\theta^* = (X_b^T X_b)^{-1} X_b^T y$. Tuy nhiên trong thực tế triển khai thư viện, `scikit-learn` không nghịch đảo ma trận trực tiếp vì chi phí tính toán lớn và dễ gặp bất ổn định số học khi ma trận điều kiện kém (ill-conditioned). Thay vào đó, Scikit-learn sử dụng giải pháp **phân rã giá trị kỳ dị (SVD - Singular Value Decomposition)** qua LAPACK (`scipy.linalg.lstsq`) để tìm nghiệm bình phương tối thiểu một cách ổn định nhất.

In [4]:
# Huấn luyện mô hình LinearRegression bằng Scikit-Learn
model = LinearRegression()
model.fit(X_train, y_train)

b_sklearn = model.intercept_
w_sklearn = model.coef_

print(f"--- KẾT QUẢ TỪ SCIKIT-LEARN ---")
print(f"model.intercept_ : {b_sklearn:.6f}")
print("model.coef_:")
for f_name, w_val in zip(features, w_sklearn):
    print(f"  {f_name:25s}: {w_val:12.6f}")

--- KẾT QUẢ TỪ SCIKIT-LEARN ---
model.intercept_ : 14.355105
model.coef_:
  fixed acidity            :     0.023085
  volatile acidity         :    -1.001304
  citric acid              :    -0.140821
  residual sugar           :     0.006564
  chlorides                :    -1.806503
  free sulfur dioxide      :     0.005627
  total sulfur dioxide     :    -0.003644
  density                  :   -10.351594
  pH                       :    -0.393688
  sulphates                :     0.841172
  alcohol                  :     0.281890


---
## 5. So sánh đối chiếu & Thực nghiệm số học (Numerical Verification)

Ta đối chiếu độ chính xác giữa kết quả giải tích (Normal Equation thuần túy) và Scikit-Learn:
1. Độ chênh lệch giữa các hệ số $\theta$ (Weights và Intercept).
2. Độ chênh lệch giữa các giá trị dự đoán $\hat{y}$ trên tập kiểm thử (Test set).

In [5]:
# So sánh độ chênh lệch giữa các hệ số tham số
max_weight_discrepancy = float(np.max(np.abs(w_math - w_sklearn)))
intercept_discrepancy = float(abs(b_math - b_sklearn))

print(f"Sai lệch tuyệt đối của Intercept |b_math - b_sklearn|: {intercept_discrepancy:.2e}")
print(f"Sai lệch tuyệt đối cực đại của Weights max|w_math - w_sklearn|: {max_weight_discrepancy:.2e}")

# So sánh dự đoán trên tập Test
y_pred_math_test = X_b_test @ theta_optimal
y_pred_sklearn_test = model.predict(X_test)

max_prediction_discrepancy = float(np.max(np.abs(y_pred_math_test - y_pred_sklearn_test)))
print(f"Sai lệch dự đoán cực đại trên tập Test: {max_prediction_discrepancy:.2e}")

# Bảng so sánh 7 mẫu đầu tiên
comparison_sample_df = pd.DataFrame({
    'Actual Quality': y_test[:7],
    'Math Predict (Normal Eq)': y_pred_math_test[:7],
    'Sklearn Predict': y_pred_sklearn_test[:7],
    'Absolute Diff': np.abs(y_pred_math_test[:7] - y_pred_sklearn_test[:7])
})
print("\n--- SO SÁNH 7 MẪU DỰ ĐOÁN ĐẦU TIÊN ---")
print(comparison_sample_df.to_string(index=False))

Sai lệch tuyệt đối của Intercept |b_math - b_sklearn|: 6.02e-09
Sai lệch tuyệt đối cực đại của Weights max|w_math - w_sklearn|: 5.10e-09
Sai lệch dự đoán cực đại trên tập Test: 1.08e-09

--- SO SÁNH 7 MẪU DỰ ĐOÁN ĐẦU TIÊN ---
 Actual Quality  Math Predict (Normal Eq)  Sklearn Predict  Absolute Diff
              6                  5.346664         5.346664       0.000000
              5                  5.056313         5.056313       0.000000
              6                  5.664470         5.664470       0.000000
              5                  5.464515         5.464515       0.000000
              6                  5.725185         5.725185       0.000000
              5                  5.279287         5.279287       0.000000
              5                  5.034217         5.034217       0.000000


In [6]:
# Bảng chi tiết hệ số hồi quy theo thứ tự 11 đặc trưng
weights_df = pd.DataFrame({
    'No.': range(1, len(features) + 1),
    'Feature': features,
    'Math Weight (w_j)': w_math,
    'Sklearn Coef (w_j)': w_sklearn,
    'Diff': np.abs(w_math - w_sklearn)
})

print("\n--- BẢNG CHI TIẾT HỆ SỐ HỒI QUY CHO SLIDE 5 ---")
print(weights_df.to_string(index=False))
print(f"\nHệ số chặn (Intercept): Math b* = {b_math:.8f} | Sklearn intercept = {b_sklearn:.8f}")


--- BẢNG CHI TIẾT HỆ SỐ HỒI QUY CHO SLIDE 5 ---
 No.              Feature  Math Weight (w_j)  Sklearn Coef (w_j)     Diff
   1        fixed acidity           0.023085            0.023085 0.000000
   2     volatile acidity          -1.001304           -1.001304 0.000000
   3          citric acid          -0.140821           -0.140821 0.000000
   4       residual sugar           0.006564            0.006564 0.000000
   5            chlorides          -1.806503           -1.806503 0.000000
   6  free sulfur dioxide           0.005627            0.005627 0.000000
   7 total sulfur dioxide          -0.003644           -0.003644 0.000000
   8              density         -10.351594          -10.351594 0.000000
   9                   pH          -0.393688           -0.393688 0.000000
  10            sulphates           0.841172            0.841172 0.000000
  11              alcohol           0.281890            0.281890 0.000000

Hệ số chặn (Intercept): Math b* = 14.35510520 | Sklearn interc

---
## 6. Đánh giá sai số & Hàm mục tiêu (Metrics Evaluation)

Hàm mục tiêu cốt lõi của bài toán là cực tiểu hóa hàm mất mát MSE. Dưới đây là giá trị của hàm mất mát và các thước đo liên quan:
- **Mean Squared Error (MSE):** Giá trị nhỏ nhất của hàm mất mát đạt được.
- **Root Mean Squared Error (RMSE):** Cùng đơn vị đo với điểm chất lượng (thang điểm 0–10).
- **Mean Absolute Error (MAE):** Trung bình độ lệch tuyệt đối giữa dự đoán và thực tế.
- **Hệ số xác định ($R^2$):** Tỷ lệ phương sai của biến mục tiêu được giải thích bởi mô hình tuyến tính.

In [7]:
y_pred_train = model.predict(X_train)
y_pred_test = model.predict(X_test)

train_mse = mean_squared_error(y_train, y_pred_train)
test_mse = mean_squared_error(y_test, y_pred_test)
test_rmse = np.sqrt(test_mse)
test_mae = mean_absolute_error(y_test, y_pred_test)
test_r2 = r2_score(y_test, y_pred_test)

metrics_summary = {
    'Train MSE (J_min)': train_mse,
    'Test MSE': test_mse,
    'Test RMSE': test_rmse,
    'Test MAE': test_mae,
    'Test R2 Score': test_r2
}

print("--- CHỈ SỐ ĐÁNH GIÁ MÔ HÌNH ---")
for m_name, m_val in metrics_summary.items():
    print(f"  {m_name:20s}: {m_val:.6f}")

--- CHỈ SỐ ĐÁNH GIÁ MÔ HÌNH ---
  Train MSE (J_min)   : 0.424191
  Test MSE            : 0.390025
  Test RMSE           : 0.624520
  Test MAE            : 0.503530
  Test R2 Score       : 0.403180


---
## 7. Xuất tham số bàn giao (Export Artifacts)

Lưu toàn bộ các tham số toán học và ánh xạ Scikit-learn ra file `reports/mathematical_parameters.json` để phục vụ biên soạn Slide 5 và phối hợp với các thành viên khác.

In [8]:
# Xuất kết quả ra file JSON với tên trường rõ ràng, chính xác
output_data = {
    "mathematical_foundation": {
        "description": "Nền tảng toán học mô hình Multiple Linear Regression cho dự án Wine Quality",
        "author": "Dao (Member 3)",
        "features": features,
        "intercept": float(b_math),
        "weights": {f: float(w) for f, w in zip(features, w_math)},
        "metrics": {
            "train_mse": float(train_mse),
            "test_mse": float(test_mse),
            "test_rmse": float(test_rmse),
            "test_mae": float(test_mae),
            "test_r2": float(test_r2)
        },
        "max_weight_discrepancy": max_weight_discrepancy,
        "intercept_discrepancy": intercept_discrepancy,
        "max_prediction_discrepancy": max_prediction_discrepancy
    }
}

json_path = REPORTS_DIR / "mathematical_parameters.json"
with open(json_path, 'w', encoding='utf-8') as f:
    json.dump(output_data, f, indent=4, ensure_ascii=False)

print(f"Đã xuất báo cáo toán học thành công tại: {json_path}")

Đã xuất báo cáo toán học thành công tại: C:\Users\Laptop\Downloads\New folder (2)\repo\reports\mathematical_parameters.json
